In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

base = Path.cwd()

if not (base / "data" / "raw").exists():
    base = base.parent

raw_dir = base / "data" / "raw"

columns = [
    "gameid",
    "date",
    "league",
    "teamname",
    "teamid",
    "position",
    "result",
    "golddiffat15",
    "dragons",
    "towers"
]

frames = []

for file in sorted(raw_dir.glob("*.csv")):
    print("Loading:", file.name)

    header = pd.read_csv(file, nrows=0).columns
    available = [c for c in columns if c in header]

    frame = pd.read_csv(
        file,
        usecols=available,
        low_memory=False
    )

    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

df["date"] = pd.to_datetime(
    df["date"], utc=True
)

teams = df[df["position"] == "team"].copy()

print("Total team records:", len(teams))
print("Total unique game IDs:", teams["gameid"].nunique())

Loading: 2017_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2018_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2019_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2020_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2021_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2022_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2023_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2024_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2025_LoL_esports_match_data_from_OraclesElixir.csv
Loading: 2026_LoL_esports_match_data_from_OraclesElixir.csv
Total team records: 192182
Total unique game IDs: 96091


In [2]:
teams = teams.dropna(
    subset=["gameid", "date", "teamname", "result"]
).copy()

teams["match_key"] = (
    teams["date"].astype(str)
    + "|"
    + teams["league"].astype(str)
    + "|"
    + teams["gameid"].astype(str)
)

# Remove exact repeated team records
teams = teams.drop_duplicates(
    subset=["match_key", "teamname"]
)

checks = teams.groupby("match_key").agg(
    n_teams=("teamname", "nunique"),
    n_rows=("teamname", "size"),
    wins=("result", "sum")
)

valid_keys = checks.index[
    (checks["n_teams"] == 2)
    & (checks["n_rows"] == 2)
    & (checks["wins"] == 1)
]

teams = teams[
    teams["match_key"].isin(valid_keys)
].copy()

teams = teams.sort_values(
    ["date", "match_key", "teamname"]
)

teams["team_index"] = (
    teams.groupby("match_key").cumcount()
)

a = teams[teams["team_index"] == 0]
b = teams[teams["team_index"] == 1]

matches = a.merge(
    b,
    on="match_key",
    suffixes=("_a", "_b"),
    validate="one_to_one"
)

matches = matches.rename(
    columns={
        "date_a": "date",
        "league_a": "league",
        "teamname_a": "team_a",
        "teamname_b": "team_b",
        "result_a": "winner_a"
    }
)

matches = matches.sort_values(
    ["date", "match_key"]
).reset_index(drop=True)

print("Valid games:", len(matches))

display(
    matches[
        ["date", "league",
         "team_a", "team_b", "winner_a"]
    ].head()
)

Valid games: 96075


,date,league,team_a,team_b,winner_a
0,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0
1,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0
2,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0
3,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1
4,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0


In [26]:
teams.groupby("league").size()

league
AC        38
AL      1936
AM       480
AOL       74
ASCI     300
        ... 
VCS     5052
VL       550
WLDs    2352
WLRQ      12
WSCI     170
Length: 124, dtype: int64

In [29]:
print(teams["date"].head())
print(teams["date"].dtype)
print(teams["date"].is_monotonic_increasing)

11   2017-01-10 15:44:06+00:00
10   2017-01-10 15:44:06+00:00
22   2017-01-10 16:21:23+00:00
23   2017-01-10 16:21:23+00:00
34   2017-01-10 17:01:50+00:00
Name: date, dtype: datetime64[us, UTC]
datetime64[us, UTC]
True


In [34]:
for game_id, group in teams.groupby("gameid"):
    row_1 = group.iloc[0]
    row_2 = group.iloc[1]
    print(row_1['teamname'], row_1['result'], row_2['teamname'], row_2['result'])

    break

Invictus Gaming Young 1 Royal Club 0


In [ ]:
match_records = []

for game_id, group in teams.groupby("gameid"):
    if len(group) != 2:
        continue

    row_1 = group.iloc[0]
    row_2 = group.iloc[1]
    
    if row_1['result'] not in [0,1] or row_2['result'] not in [0,1] or row_1['result'] + row_2['result'] != 1:
        continue
    else:
        match_record = {
            "game_id": game_id,
            "date": row_1['date'],
            "team_a": row_1['teamname'], 
            "team_b": row_2['teamname'],
            "result_a" : row_1['result']
        }
        match_records.append(match_record)


In [3]:
def expected_score(rating_a, rating_b):
    return 1/(1+10**((rating_b - rating_a)/400))


In [4]:
def update_elo(rating_a, rating_b, result_a, k = 24):
    winrate_a = expected_score(rating_a, rating_b) 
    rating_a_new = rating_a + k * (result_a - winrate_a)
    rating_b_new = rating_b + k * ((1-result_a) - (1-winrate_a))
    return rating_a_new, rating_b_new

In [5]:
elo = {}

INITIAL_ELO = 1500 

if 'T1' not in elo:
    elo['T1'] = INITIAL_ELO
print(elo)

elo['T1'] = 1524

print(elo['T1'])



{'T1': 1500}
1524


In [6]:
matches = [
    ("T1", "Gen.G", 1),
    ("T1", "DK", 0),
    ("Gen.G", "DK", 1)
]


In [11]:
# match = {match:1, "team_a":"t1", "team_b":"geng", "pre_elo_a":1500, "pre_elo_b":1500, "match_result":1}
# match_features.append(elo)
elo = {}
match_features = []

for match in matches:
    if match[0] not in elo:
        elo[match[0]] = INITIAL_ELO

    if match[1] not in elo:
        elo[match[1]] = INITIAL_ELO
        
    elo_team_a = elo[match[0]]
    elo_team_b = elo[match[1]]
    match_record = {
        "team_a": match[0],
        "team_b": match[1],
        "pre_elo_a": elo_team_a,
        "pre_elo_b": elo_team_b,
        "match_result": match[2]
    }
    match_features.append(match_record)
    elo[match[0]], elo[match[1]] = update_elo(match_record['pre_elo_a'], match_record['pre_elo_b'], match_record["match_result"], 24)

In [12]:
features_df = pd.DataFrame(match_features)

In [13]:
print(features_df.head(5))
print(features_df.shape)
print(features_df.dtypes)

  team_a team_b  pre_elo_a    pre_elo_b  match_result
0     T1  Gen.G     1500.0  1500.000000             1
1     T1     DK     1512.0  1500.000000             0
2  Gen.G     DK     1488.0  1512.414301             1
(3, 5)
team_a              str
team_b              str
pre_elo_a       float64
pre_elo_b       float64
match_result      int64
dtype: object


In [14]:
features_df['elo_diff'] = features_df['pre_elo_a'] - features_df['pre_elo_b']
print(features_df.head())

  team_a team_b  pre_elo_a    pre_elo_b  match_result   elo_diff
0     T1  Gen.G     1500.0  1500.000000             1   0.000000
1     T1     DK     1512.0  1500.000000             0  12.000000
2  Gen.G     DK     1488.0  1512.414301             1 -24.414301


In [16]:
X = features_df[['elo_diff']]
y = features_df['match_result']
print(X.shape, y.shape, X.head(), y.head())


(3, 1) (3,)     elo_diff
0   0.000000
1  12.000000
2 -24.414301 0    1
1    0
2    1
Name: match_result, dtype: int64


In [20]:
X

,elo_diff
0,0.000000
1,12.000000
2,-24.414301


In [17]:
from sklearn.linear_model import LogisticRegression


In [ ]:
model = LogisticRegression()
clf = model.fit(X,y)
print(clf.intercept_, clf.coef_)

[3.10104862] [[-0.51683237]]


In [25]:
test_match = [[100]]
clf.predict(test_match)
clf.predict_proba(test_match)

c:\Users\USER\lol_worlds_predictor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\USER\lol_worlds_predictor\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


array([[1.0000000e+00, 7.9620254e-22]])